# One-region noise parameter simulation


This notebook compares the excitatory output `E` under a constant low baseline input while varying the two noise parameters of the Zerlaut model. It follows the one-region workflow of `01b`.


## Initialization



Load environment variables

<font color=#fda> NOTE: </font> Prior to loading any modules, we need to load the environment variables from the configuration files. This way, we can configure the TVB modules using environment variables.


In [ ]:
from dotenv import dotenv_values, find_dotenv
import os
import sys
from pathlib import Path


# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))


Modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
from aux.environment import resolve_directory
from aux.tvb.model import build_model
from aux.tvb.integrator import build_integrator
from aux.tvb.simulator import build_single_region_simulator
from aux.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata
from aux.tvb.plotting import plot_results


Console

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

Set main directory paths

In [ ]:

#ToDo: Cambiar el nombe de las variables de entorno
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Simulation values


In [ ]:
# 1) Simulation values.
DURATION_MS = 2000.0
DT_MS = 0.5
OUTPUT_DTYPE = "float32"
OUTPUT_STATES = ["E"]
STATE_UNITS = {"E": "kHz"}
BASELINE_KHZ = 0.005
PLOT_START_MS = 500.0  # Exclude the initial transient from the comparison.


In [ ]:
# 1.1) Noise grid. The integrator noise and its seed stay fixed.
WEIGHT_NOISE_VALUES = [0.0, 2e-4, 2e-3, 1e-2]
TAU_OU_VALUES = [0.5, 2.0, 8.0]
NOISE_SEED = 42
NOISE = {
    "name": "Additive",
    "params": {"nsig": [0, 0, 0, 0, 1]},
}


In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"


MODEL_PARAMETERS = {
    # Población y conectividad
    "N_tot": 10000,
    "p_connect_e": 0.05,
    "p_connect_i": 0.05,
    "g": 0.2,
    "K_ext_e": 400,
    "K_ext_i": 35,
    
    # Parametros de membrana
    "g_L": 10.0,
    "C_m": 200.0,
    "E_L_e": -60.0,
    "E_L_i": -65.0,

    # Parametors de adaptación
    "a_e": 4.0,
    "b_e": 60.0,
    "tau_w_e": 200.0,
    "a_i": 0.0,
    "b_i": 20.0,
    "tau_w_i": 100.0,

    # Parametros sinápticos
    "E_e": 0.0,
    "E_i": -80.0,
    "Q_e": 1.0,
    "Q_i": 5.0,
    "tau_e": 5.0,
    "tau_i": 5.0,
    
    "T": 10.0,

    # Parametro de segundo orden (no hay un motivo matemático)
    "S_i": 1, # Escala la entrada remota recibida por la población inhibidora en el modelo de segundo orden.

    # Parametros de ruido
    "weight_noise": 2e-4, #10.5
    "tau_OU": 2, #0.5
}

# Todos los parámetros son de tipo float menos lo que están en esta lista. 
INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}


The constant baseline is assigned to the model parameter `external_input_ex_ex`. `weight_noise` scales the Ornstein-Uhlenbeck contribution to the external input; `tau_OU` changes its time scale. The stochastic integrator drives only `ou_drift`, with the same random sequence in every grid cell.


## Stochastic Simulation


### Run simulation


In [ ]:
results = {}
for tau_OU in TAU_OU_VALUES:
    for weight_noise in WEIGHT_NOISE_VALUES:
        parameters = MODEL_PARAMETERS | {
            "external_input_ex_ex": BASELINE_KHZ,
            "weight_noise": weight_noise,
            "tau_OU": tau_OU,
        }
        integrator = build_integrator(DT_MS, integrator_class="HeunStochastic",
                                      noise_config=NOISE)
        integrator.noise.random_stream.seed(NOISE_SEED)
        model = build_model(
            parameters, INTEGER_PARAMETERS,
            model_class="ZerlautAdaptationFirstOrder",
            variable_of_interest=OUTPUT_STATES,
        )
        simulator = build_single_region_simulator(model, [], integrator)
        time_sim, raw = simulator.run(n_steps=round(DURATION_MS / DT_MS))[0]
        states = {"E": raw[:, model.state_variables.index("E"), 0, 0]}
        metadata = ResultMetadata(
            simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
            model=ModelMetadata(
                name=model, parameters=parameters,
                integer_parameters=list(INTEGER_PARAMETERS),
                output_variables=OUTPUT_STATES, state_units=STATE_UNITS,
            ),
            stimulus={},
        )
        results[(tau_OU, weight_noise)] = (time_sim, states, metadata)
        steady_E = states["E"][time_sim >= PLOT_START_MS]
        pprint(f"tau_OU={tau_OU:g} ms, weight_noise={weight_noise:g}: "
               f"E standard deviation={steady_E.std():.5f} kHz")


### Plot results


In [ ]:
fig, axes = plt.subplots(
    len(TAU_OU_VALUES), len(WEIGHT_NOISE_VALUES),
    figsize=(4.5 * len(WEIGHT_NOISE_VALUES), 2.8 * len(TAU_OU_VALUES)),
    constrained_layout=True,
)
plotted_axes = []
for row, tau_OU in enumerate(TAU_OU_VALUES):
    for column, weight_noise in enumerate(WEIGHT_NOISE_VALUES):
        time_sim, states, metadata = results[(tau_OU, weight_noise)]
        _, panel_axes = plot_results(
            time_sim, states, metadata, show_stimulus=False,
            plot_states=["E"], ax=axes[row, column],
        )
        panel_axes[0].set_title(f"tau_OU={tau_OU:g} ms; weight_noise={weight_noise:g}")
        plotted_axes.append(panel_axes[0])

for axis in plotted_axes:
    axis.set_xlim(PLOT_START_MS, DURATION_MS)
fig.suptitle(f"E with constant external_input_ex_ex = {BASELINE_KHZ:g} kHz")
plt.show()
